# Task 4: Text Vectorization Using TF-IDF and Similarity Analysis.
## Activity 1: Review the cleaned dataset

In [28]:
# Step 1.1 — Import pandas and load the CSV

import pandas as pd

df = pd.read_csv("../data/processed/ted_talks_cleaned.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset loaded successfully!
Shape: (4005, 24)

Columns:
['talk_id', 'title', 'speaker_1', 'all_speakers', 'occupations', 'about_speakers', 'views', 'recorded_date', 'published_date', 'event', 'native_lang', 'available_lang', 'comments', 'duration', 'topics', 'related_talks', 'url', 'description', 'transcript', 'title_clean', 'description_clean', 'transcript_clean', 'topics_clean', 'combined_text']


In [29]:
# Step 1.2 — Check the title and combined text columns
print(df.head(5))
print("\nMissing values:")
print(df.isnull().sum())

   talk_id                            title         speaker_1  \
0        1      Averting the climate crisis           Al Gore   
1       92  The best stats you've ever seen      Hans Rosling   
2        7                 Simplicity sells       David Pogue   
3       53              Greening the ghetto     Majora Carter   
4       66      Do schools kill creativity?  Sir Ken Robinson   

              all_speakers                                    occupations  \
0           {0: 'Al Gore'}                      {0: ['climate advocate']}   
1      {0: 'Hans Rosling'}  {0: ['global health expert; data visionary']}   
2       {0: 'David Pogue'}                  {0: ['technology columnist']}   
3     {0: 'Majora Carter'}    {0: ['activist for environmental justice']}   
4  {0: 'Sir Ken Robinson'}                    {0: ['author', 'educator']}   

                                      about_speakers     views recorded_date  \
0  {0: 'Nobel Laureate Al Gore focused the world’...   3523392    

In [30]:
# Check the required columns:
print("Available columns:", df.columns.tolist())

assert "title" in df.columns, "Title column not found"
assert "combined_text" in df.columns, "Combined text column not found"

Available columns: ['talk_id', 'title', 'speaker_1', 'all_speakers', 'occupations', 'about_speakers', 'views', 'recorded_date', 'published_date', 'event', 'native_lang', 'available_lang', 'comments', 'duration', 'topics', 'related_talks', 'url', 'description', 'transcript', 'title_clean', 'description_clean', 'transcript_clean', 'topics_clean', 'combined_text']


In [31]:
# Step 1.3 — Count usable records

df = df.dropna(subset=["title", "combined_text"]).copy()

df["title"] = df["title"].astype(str).str.strip()
df["combined_text"] = df["combined_text"].fillna("").astype(str).str.strip()

df = df[
    (df["title"] != "") &
    (df["combined_text"] != "")
].reset_index(drop=True)

print("Usable TED Talks:", len(df))

Usable TED Talks: 4005


## Activity 2: Implement TF-IDF vectorization

In [32]:
# Step 2.1 — Import TfidfVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

print("TF-IDF imported successfully!")

TF-IDF imported successfully!


In [33]:
# Step 2.2 — Convert text into numerical vectors
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=10000
)

tfidf_matrix = vectorizer.fit_transform(df["combined_text"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (4005, 10000)


In [34]:
# Step 2.3 — Inspect the vocabulary
feature_names = vectorizer.get_feature_names_out()

print("Total vocabulary words:", len(feature_names))
print("First 30 words:")
print(feature_names[:30])

Total vocabulary words: 10000
First 30 words:
['00' '000' '10' '100' '1000' '101' '10th' '11' '110' '11th' '12' '120'
 '125' '12th' '13' '130' '13th' '14' '140' '14th' '15' '150' '15th' '16'
 '160' '16th' '17' '170' '17th' '18']


In [35]:
# Step 2.4 — Understand the matrix dimensions
# TF-IDF matrix shape: (4005, 10000)

In [36]:
# Step 2.5 — Display a small matrix sample
sample = pd.DataFrame(
    tfidf_matrix[:3, :10].toarray(),
    columns=feature_names[:10],
    index=df["title"].iloc[:3]
)

print(sample)

                                  00       000        10       100  1000  101  \
title                                                                           
Averting the climate crisis      0.0  0.000000  0.000000  0.000000   0.0  0.0   
The best stats you've ever seen  0.0  0.008438  0.024773  0.031285   0.0  0.0   
Simplicity sells                 0.0  0.008669  0.008483  0.000000   0.0  0.0   

                                 10th        11  110  11th  
title                                                       
Averting the climate crisis       0.0  0.000000  0.0   0.0  
The best stats you've ever seen   0.0  0.000000  0.0   0.0  
Simplicity sells                  0.0  0.042348  0.0   0.0  


## Activity 3: Perform similarity analysis

In [37]:
# Step 3.1 — Import cosine similarity
from sklearn.metrics.pairwise import cosine_similarity

print("Cosine similarity imported successfully!")

Cosine similarity imported successfully!


In [38]:
# Step 3.2 — Select a TED Talk
for i, title in enumerate(df["title"].head(20)):
    print(i, title)

0 Averting the climate crisis
1 The best stats you've ever seen
2 Simplicity sells
3 Greening the ghetto
4 Do schools kill creativity?
5 Behind the design of Seattle's library
6 Letting go of God
7 Let's teach religion -- all religion -- in schools
8 A life of purpose
9 My wish: A global day of film
10 My wish: Help me stop pandemics
11 My wish: A call for open-source architecture
12 One Laptop per Child
13 The radical promise of the multi-touch interface
14 Improvising on piano, aged 14
15 An 11-year-old's magical violin
16 Simple designs to save a life
17 Organic design, inspired by nature
18 The birth of Wikipedia
19 The birth of the open-source learning revolution


In [39]:
# Choose one title from the output
selected_title = "The best stats you've ever seen"

matches = df.index[df["title"] == selected_title].tolist()

if not matches:
    print("Title not found. Check the spelling.")
else:
    talk_index = matches[0]
    print("Selected talk:", df.loc[talk_index, "title"])

Selected talk: The best stats you've ever seen


In [40]:
# Step 3.3 — Calculate similarities
similarities = cosine_similarity(
    tfidf_matrix[talk_index],
    tfidf_matrix
).flatten()

print("Number of similarity scores:", len(similarities))

Number of similarity scores: 4005


In [41]:
# Step 3.4 — Display the most similar talks
similarities[talk_index] = -1

top_indices = similarities.argsort()[::-1][:5]

for i in top_indices:
    print(df.loc[i, "title"], "->", round(similarities[i], 4))

Let my dataset change your mindset -> 0.6103
New insights on poverty -> 0.4392
The good news of the decade? We're winning the war against child mortality -> 0.3817
Global population growth, box by box -> 0.3635
Insights on HIV, in stunning data visuals -> 0.3509


## Activity 4: Generate recommendations for three talks

In [42]:
# Step 4.1 — Create the recommendation function
def recommend_talks(title, top_n=5):
    matches = df.index[df["title"] == title].tolist()

    if not matches:
        print("Talk not found:", title)
        return pd.DataFrame()

    idx = matches[0]

    scores = cosine_similarity(
        tfidf_matrix[idx], tfidf_matrix
    ).flatten()

    scores[idx] = -1

    top_idx = scores.argsort()[::-1][:top_n]

    return pd.DataFrame({
        "Selected Talk": title,
        "Recommended Talk": df.loc[top_idx, "title"].values,
        "Similarity Score": scores[top_idx].round(4)
    })

In [43]:
# Step 4.2 — Choose three real TED Talks
print(df["title"].head(50).to_string(index=False))

                       Averting the climate crisis
                   The best stats you've ever seen
                                  Simplicity sells
                               Greening the ghetto
                       Do schools kill creativity?
            Behind the design of Seattle's library
                                 Letting go of God
Let's teach religion -- all religion -- in schools
                                 A life of purpose
                     My wish: A global day of film
                   My wish: Help me stop pandemics
      My wish: A call for open-source architecture
                              One Laptop per Child
  The radical promise of the multi-touch interface
                     Improvising on piano, aged 14
                   An 11-year-old's magical violin
                     Simple designs to save a life
                Organic design, inspired by nature
                            The birth of Wikipedia
  The birth of the open-source 

In [44]:
# Choose three different titles from the output.
selected_titles = df["title"].iloc[[0, 10, 20]].tolist()

print(selected_titles)

['Averting the climate crisis', 'My wish: Help me stop pandemics', 'Nerdcore comedy']


In [45]:
# Step 4.3 — Generate recommendations for all three
results = []

for title in selected_titles:
    result = recommend_talks(title, top_n=5)
    if not result.empty:
        results.append(result)

if results:
    all_recommendations = pd.concat(results, ignore_index=True)
    print(all_recommendations.to_string(index=False))

                  Selected Talk                                         Recommended Talk  Similarity Score
    Averting the climate crisis                       New thinking on the climate crisis            0.3110
    Averting the climate crisis               A climate solution where all sides can win            0.3103
    Averting the climate crisis          How we could change the planet's climate future            0.2978
    Averting the climate crisis Can we stop climate change by removing CO2 from the air?            0.2882
    Averting the climate crisis    The disarming case to act right now on climate change            0.2828
My wish: Help me stop pandemics                                    The case for optimism            0.4669
My wish: Help me stop pandemics             A global pandemic calls for global solutions            0.4538
My wish: Help me stop pandemics                                Lessons from the 1918 flu            0.2818
My wish: Help me stop pandemics      

In [46]:
# Step 4.4 — Save your recommendation results
if results:
    all_recommendations.to_csv(
        "../data/processed/task4_recommendations.csv",
        index=False
    )
    print("Recommendation results saved successfully!")

Recommendation results saved successfully!


In [47]:
# Check the actual content of the talks
for title in selected_titles:
    print("\nSELECTED TALK:", title)

    idx = df.index[df["title"] == title][0]
    print(df.loc[idx, "combined_text"][:500])

    recs = all_recommendations[
        all_recommendations["Selected Talk"] == title
    ]

    for _, row in recs.iterrows():
        rec_idx = df.index[
            df["title"] == row["Recommended Talk"]
        ][0]

        print("\nRECOMMENDED:", row["Recommended Talk"])
        print("Score:", row["Similarity Score"])
        print(df.loc[rec_idx, "combined_text"][:300])


SELECTED TALK: Averting the climate crisis
averting the climate crisis alternative energy cars climate change culture environment global issues science sustainability technology with the same humor and humanity he exuded in an inconvenient truth al gore spells out 15 ways that individuals can address climate change immediately from buying a hybrid to inventing a new hotter brand name for global warming thank you so much chris and it's truly a great honor to have the opportunity to come to this stage twice i'm extremely grateful i have be

RECOMMENDED: New thinking on the climate crisis
Score: 0.311
new thinking on the climate crisis activism climate change global issues science in this brand-new slideshow premiering on ted com al gore presents evidence that the pace of climate change may be even worse than scientists recently predicted he challenges us to act i have given the slide show that i

RECOMMENDED: A climate solution where all sides can win
Score: 0.3103
a climate solution wh

# Activity 5: Analyze Limitations

The purpose of this activity is to analyze the relevance of the recommendations generated using TF-IDF and cosine similarity and identify the limitations of the approach.


In [48]:
# Check the limitations using code
print(all_recommendations.groupby("Selected Talk")["Similarity Score"].agg(
    ["min", "max", "mean"]
).round(4))

                                    min     max    mean
Selected Talk                                          
Averting the climate crisis      0.2828  0.3110  0.2980
My wish: Help me stop pandemics  0.2236  0.4669  0.3411
Nerdcore comedy                  0.3953  0.4436  0.4186


## Observations and Limitations

### 1. Observation: Climate-Related Talks
For **“Averting the climate crisis,”** the similarity scores range from **0.2828 to 0.3110**, with an average of **0.2980**. The recommended talk “New thinking on the climate crisis” received the highest score of **0.3110**. The recommendations show overlapping climate-related vocabulary.

### 2. Observation: Pandemic-Related Talks
For **“My wish: Help me stop pandemics,”** the scores range from **0.2236 to 0.4669**, with an average of **0.3411**. “The case for optimism” received the highest score of **0.4669**, while “A global pandemic calls for global solutions” received **0.4538**. This shows that TF-IDF may rank talks highly because of shared words, although their actual relevance needs to be checked by reading their content.

### 3. Observation: Comedy-Related Talks
For **“Nerdcore comedy,”** the scores range from **0.3953 to 0.4436**, with an average of **0.4186**. “All things are Moleeds” received the highest score of **0.4436**, followed by “A comic sendup of TED2006” with **0.4300**. These scores indicate similarities in the text, but the content should be reviewed to confirm topic relevance.

### Limitations

1. **Similar words do not always mean similar topics:** Talks may share vocabulary but discuss different ideas.
2. **Different words can express similar meanings:** TF-IDF may give lower scores to talks that discuss similar concepts using different vocabulary.
3. **Limited understanding of context:** TF-IDF does not fully understand meaning, context, or sarcasm.
4. **Similarity does not guarantee user satisfaction:** A high score does not mean that every user will like the recommended talk.
5. **Dependence on text quality:** Missing or limited text can affect the quality of the recommendations.

### Conclusion

TF-IDF and cosine similarity successfully converted TED Talk text into numerical representations and ranked talks according to their textual similarity. In my results, the average similarity scores were 0.2980 for “Averting the climate crisis,” 0.3411 for “My wish: Help me stop pandemics,” and 0.4186 for “Nerdcore comedy.” However, high similarity scores do not guarantee semantic relevance or user satisfaction. Manual review is necessary to evaluate the recommendations more effectively.